In [1]:
from IPython.display import HTML, display
import ipywidgets as widgets
import numpy as np

In [2]:
# -------------------------------------------------------------------
# 1. ELECTRICAL & GEOMETRY CALCULATION ENGINE
# -------------------------------------------------------------------
def calculate_miller_capacitance(
    gbw_mhz, gm1_us, ratio_cm1, mmin, mmax, l_fixed_um, square_layout
):
    gbw_hz = gbw_mhz * 1e6
    gm1_s = gm1_us * 1e-6

    # Step 1: Total Cc calculation
    cc_total_f = gm1_s / (2 * np.pi * gbw_hz)
    cc_total_pf = cc_total_f * 1e12
    cc_total_ff = cc_total_pf * 1e3

    # Step 2: CM1 & CM2 split
    cm1_ff = cc_total_ff * ratio_cm1
    cm2_ff = cc_total_ff * (1.0 - ratio_cm1)

    # Step 3: Area capacitance density (fF/um^2)
    base_cap = 0.372 if mmin == 1 else 0.305
    areacap = base_cap + (mmax - mmin) * 0.305

    # Step 4: Physical dimensions calculation
    if square_layout:
        w_cm1 = l_cm1 = np.sqrt(cm1_ff / areacap) if areacap > 0 else 0
        w_cm2 = l_cm2 = np.sqrt(cm2_ff / areacap) if areacap > 0 else 0
    else:
        l_cm1 = l_fixed_um
        w_cm1 = cm1_ff / (areacap * l_cm1) if (areacap * l_cm1) > 0 else 0
        l_cm2 = l_fixed_um
        w_cm2 = cm2_ff / (areacap * l_cm2) if (areacap * l_cm2) > 0 else 0

    return {
        "cc_total_ff": cc_total_ff,
        "cc_total_pf": cc_total_pf,
        "cm1_ff": cm1_ff,
        "cm2_ff": cm2_ff,
        "areacap": areacap,
        "w_cm1": w_cm1,
        "l_cm1": l_cm1,
        "w_cm2": w_cm2,
        "l_cm2": l_cm2,
    }

In [3]:
# -------------------------------------------------------------------
# 2. WIDGET CONTROLS
# -------------------------------------------------------------------
style = {"description_width": "180px"}
layout = widgets.Layout(width="450px")

w_gbw = widgets.FloatText(
    value=13.0, description="Target GBW (MHz):", style=style, layout=layout
)
w_gm1 = widgets.FloatText(
    value=82.0, description="Input gm1 (µS):", style=style, layout=layout
)
w_ratio = widgets.FloatSlider(
    value=0.5,
    min=0.1,
    max=0.9,
    step=0.01,
    description="CM1 Ratio:",
    style=style,
    layout=layout,
)
w_mmin = widgets.IntSlider(
    value=1,
    min=1,
    max=4,
    step=1,
    description="mmin (Bottom Metal):",
    style=style,
    layout=layout,
)
w_mmax = widgets.IntSlider(
    value=4,
    min=2,
    max=5,
    step=1,
    description="mmax (Top Metal):",
    style=style,
    layout=layout,
)
w_square = widgets.Checkbox(
    value=True,
    description="Square Layout (W = L)",
    style=style,
    layout=layout,
)
w_l_fixed = widgets.FloatSlider(
    value=10.0,
    min=1.0,
    max=50.0,
    step=0.5,
    description="Fixed Length L (µm):",
    style=style,
    layout=layout,
)

out = widgets.Output()

In [4]:
# -------------------------------------------------------------------
# 3. UPDATE & RENDER FUNCTIONS
# -------------------------------------------------------------------
def update_calculator(change=None):
    # Validasi urutan metal stack
    if w_mmax.value <= w_mmin.value:
        w_mmax.value = w_mmin.value + 1

    res = calculate_miller_capacitance(
        gbw_mhz=w_gbw.value,
        gm1_us=w_gm1.value,
        ratio_cm1=w_ratio.value,
        mmin=w_mmin.value,
        mmax=w_mmax.value,
        l_fixed_um=w_l_fixed.value,
        square_layout=w_square.value,
    )

    with out:
        out.clear_output(wait=True)

        html_summary = f"""
        <div style="background-color: #1e1e1e; color: #ffffff; padding: 15px; border-radius: 8px; font-family: monospace; min-width: 450px;">
            <h4 style="color: #4CAF50; margin-top:0;">--> Hasil Estimasi Miller Capacitor (cap_cmomf)</h4>
            <b>Total Compensation (C_C):</b> {res['cc_total_pf']:.3f} pF ({res['cc_total_ff']:.2f} fF)<br>
            <b>Target C_M1 (PMOS side):</b> <span style="color: #4DABF7;">{res['cm1_ff']:.2f} fF</span><br>
            <b>Target C_M2 (NMOS side):</b> <span style="color: #4DABF7;">{res['cm2_ff']:.2f} fF</span><br>
            <b>Capacitance Density (areacap):</b> {res['areacap']:.3f} fF/µm²<br>
            <hr style="border: 0.5px solid #444;">
            <b>C_M1 Dimensions:</b> <span style="color: #FFD700; font-size: 15px;">W = {res['w_cm1']:.2f} µm, L = {res['l_cm1']:.2f} µm</span><br>
            <b>C_M2 Dimensions:</b> <span style="color: #69DB7C; font-size: 15px;">W = {res['w_cm2']:.2f} µm, L = {res['l_cm2']:.2f} µm</span><br>
        </div>
        """
        display(HTML(html_summary))


# Connect observers
for w in [w_gbw, w_gm1, w_ratio, w_mmin, w_mmax, w_square, w_l_fixed]:
    w.observe(update_calculator, names="value")

In [5]:
# -------------------------------------------------------------------
# 4. RENDER DASHBOARD
# -------------------------------------------------------------------
input_box = widgets.VBox(
    [
        widgets.HTML("<h3>Parameters & Circuit Targets</h3>"),
        w_gbw,
        w_gm1,
        w_ratio,
        w_mmin,
        w_mmax,
        w_square,
        w_l_fixed,
    ]
)

display(widgets.HBox([input_box, out]))

# Trigget initial calculation
update_calculator()